# Frame Cache Layer

**EvoIR Stage 1 — Cosine similarity-based frame caching for video SR**

Key features:
- 64×64 thumbnail comparison via cosine similarity
- Configurable similarity threshold
- LRU cache with bounded size
- Cache statistics logging

---

In [ ]:
import torch
import torch.nn.functional as F
from collections import OrderedDict
from typing import Optional, Tuple, Dict
import time

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## Frame Cache Implementation

In [ ]:
class FrameCache:
    """
    Frame similarity cache for adaptive video processing.
    
    Compares incoming frames against cached results using cosine similarity
    of 64×64 thumbnails. If similar enough (cache hit), returns cached result
    to skip redundant AFLB processing.
    
    Architecture (from HoCVid pipeline diagram):
    Degraded Frame → Thumbnail (64×64) → Cosine Similarity Check
       └→ Cache Hit:  Return cached restored frame (bypass Stage 1)
       └→ Cache Miss: Process through AFLB → Store result in cache
    
    Args:
        cache_size: Maximum number of cached frames (LRU eviction)
        similarity_threshold: Cosine similarity threshold for cache hit (0-1)
        thumbnail_size: Size of comparison thumbnail (default: 64)
    """
    
    def __init__(
        self,
        cache_size: int = 32,
        similarity_threshold: float = 0.95,
        thumbnail_size: int = 64
    ):
        self.cache_size = cache_size
        self.similarity_threshold = similarity_threshold
        self.thumbnail_size = thumbnail_size
        
        # LRU cache: key -> (thumbnail_vector, cached_output)
        self.cache: OrderedDict[int, Tuple[torch.Tensor, torch.Tensor]] = OrderedDict()
        self._next_key = 0
        
        # Statistics
        self.stats = {
            'total_queries': 0,
            'cache_hits': 0,
            'cache_misses': 0,
            'time_saved_ms': 0.0,
        }
    
    def _make_thumbnail(self, frame: torch.Tensor) -> torch.Tensor:
        """
        Create normalized thumbnail vector for comparison.
        
        Args:
            frame: Input frame (C, H, W) or (B, C, H, W)
        Returns:
            Flattened, normalized thumbnail vector
        """
        if frame.dim() == 4:
            frame = frame[0]  # Take first in batch
        
        # Resize to thumbnail
        thumb = F.interpolate(
            frame.unsqueeze(0).float(),
            size=(self.thumbnail_size, self.thumbnail_size),
            mode='bilinear',
            align_corners=False
        )
        
        # Flatten and L2-normalize
        thumb_vec = thumb.reshape(-1)
        thumb_vec = F.normalize(thumb_vec, dim=0)
        
        return thumb_vec
    
    def _cosine_similarity(self, vec1: torch.Tensor, vec2: torch.Tensor) -> float:
        """Compute cosine similarity between two normalized vectors."""
        return torch.dot(vec1, vec2).item()
    
    def query(
        self, frame: torch.Tensor
    ) -> Tuple[bool, Optional[torch.Tensor], float]:
        """
        Check if a similar frame exists in cache.
        
        Args:
            frame: Input frame tensor (C, H, W) or (B, C, H, W)
        Returns:
            Tuple of:
            - hit: Whether cache hit occurred
            - result: Cached output if hit, None if miss
            - similarity: Best similarity score found
        """
        self.stats['total_queries'] += 1
        thumb = self._make_thumbnail(frame)
        
        best_sim = -1.0
        best_key = None
        
        for key, (cached_thumb, _) in self.cache.items():
            sim = self._cosine_similarity(thumb, cached_thumb.to(thumb.device))
            if sim > best_sim:
                best_sim = sim
                best_key = key
        
        if best_sim >= self.similarity_threshold and best_key is not None:
            # Cache HIT
            self.stats['cache_hits'] += 1
            # Move to end (most recently used)
            self.cache.move_to_end(best_key)
            _, cached_result = self.cache[best_key]
            return True, cached_result, best_sim
        else:
            # Cache MISS
            self.stats['cache_misses'] += 1
            return False, None, best_sim
    
    def store(
        self, frame: torch.Tensor, result: torch.Tensor
    ) -> int:
        """
        Store a processed frame result in cache.
        
        Args:
            frame: Original input frame
            result: Processed output to cache
        Returns:
            Cache key assigned
        """
        thumb = self._make_thumbnail(frame)
        key = self._next_key
        self._next_key += 1
        
        # Store thumbnail and result (on CPU to save GPU memory)
        self.cache[key] = (thumb.detach().cpu().clone(), result.detach().cpu().clone())
        
        # LRU eviction
        while len(self.cache) > self.cache_size:
            self.cache.popitem(last=False)  # Remove oldest
        
        return key
    
    def process_frame(
        self,
        frame: torch.Tensor,
        process_fn,
        **kwargs
    ) -> Tuple[torch.Tensor, bool]:
        """
        Unified cache-aware processing pipeline.
        
        1. Check cache for similar frame
        2. On hit: return cached result
        3. On miss: run process_fn, store result, return
        
        Args:
            frame: Input frame tensor
            process_fn: Function to process frame on cache miss (model forward)
            **kwargs: Additional args for process_fn
        Returns:
            Tuple of (result, was_cache_hit)
        """
        hit, cached_result, sim = self.query(frame)
        
        if hit:
            return cached_result.to(frame.device), True
        else:
            # Process and cache
            result = process_fn(frame, **kwargs)
            self.store(frame, result)
            return result, False
    
    def get_stats(self) -> Dict[str, float]:
        """Get cache performance statistics."""
        total = self.stats['total_queries']
        hit_rate = self.stats['cache_hits'] / total if total > 0 else 0
        return {
            **self.stats,
            'hit_rate': hit_rate,
            'cache_occupancy': len(self.cache),
            'cache_capacity': self.cache_size,
        }
    
    def clear(self):
        """Clear cache and reset statistics."""
        self.cache.clear()
        self.stats = {
            'total_queries': 0,
            'cache_hits': 0,
            'cache_misses': 0,
            'time_saved_ms': 0.0,
        }
    
    def __repr__(self):
        stats = self.get_stats()
        return (
            f"FrameCache(size={self.cache_size}, threshold={self.similarity_threshold}, "
            f"occupancy={stats['cache_occupancy']}, "
            f"hit_rate={stats['hit_rate']:.1%})"
        )

print('FrameCache defined')

## Verification Tests

In [ ]:
print('=== Test 1: Basic cache miss/store ===')
cache = FrameCache(cache_size=8, similarity_threshold=0.95)
frame1 = torch.randn(3, 128, 128)
hit, result, sim = cache.query(frame1)
print(f'  Empty cache query: hit={hit}, sim={sim:.4f}')
assert not hit and result is None
cache.store(frame1, torch.randn(3, 256, 256))
print(f'  After store: {cache}')
print('  PASSED')

print('\n=== Test 2: Cache hit with identical frame ===')
hit, result, sim = cache.query(frame1)
print(f'  Same frame query: hit={hit}, sim={sim:.4f}')
assert hit and result is not None and sim > 0.999
print('  PASSED')

print('\n=== Test 3: Cache miss with different frame ===')
frame2 = torch.randn(3, 128, 128)  # Totally different
hit, result, sim = cache.query(frame2)
print(f'  Different frame query: hit={hit}, sim={sim:.4f}')
assert not hit
print('  PASSED')

print('\n=== Test 4: Cache hit with similar frame (small noise) ===')
frame3 = frame1 + torch.randn_like(frame1) * 0.01  # Tiny perturbation
hit, result, sim = cache.query(frame3)
print(f'  Similar frame query: hit={hit}, sim={sim:.4f}')
# Should be a hit since similarity > threshold
assert hit, f'Expected hit but sim={sim:.4f}'
print('  PASSED')

print('\n=== Test 5: LRU eviction ===')
small_cache = FrameCache(cache_size=3, similarity_threshold=0.95)
for i in range(5):
    f = torch.randn(3, 64, 64) * (i + 1)  # Different frames
    small_cache.store(f, torch.randn(3, 128, 128))
print(f'  After storing 5 frames in cache_size=3: occupancy={len(small_cache.cache)}')
assert len(small_cache.cache) == 3, 'LRU eviction failed'
print('  PASSED')

print('\n=== Test 6: process_frame unified API ===')
cache2 = FrameCache(cache_size=16, similarity_threshold=0.95)

# Mock processing function
def mock_model(frame):
    return frame * 2  # Simple transformation

result1, was_hit = cache2.process_frame(frame1, mock_model)
print(f'  First call: hit={was_hit}')
assert not was_hit

result2, was_hit = cache2.process_frame(frame1, mock_model)
print(f'  Second call (same frame): hit={was_hit}')
assert was_hit

stats = cache2.get_stats()
print(f'  Stats: {stats}')
assert stats['cache_hits'] == 1
assert stats['cache_misses'] == 1
assert abs(stats['hit_rate'] - 0.5) < 0.01
print('  PASSED')

print('\n=== Test 7: Video sequence simulation ===')
cache3 = FrameCache(cache_size=16, similarity_threshold=0.90)
base_frame = torch.randn(3, 128, 128)
hits, misses = 0, 0

for i in range(30):
    if i % 10 == 0:  # Scene change every 10 frames
        base_frame = torch.randn(3, 128, 128)
    # Add temporal noise (small between-frame changes)
    frame = base_frame + torch.randn_like(base_frame) * 0.05
    result, was_hit = cache3.process_frame(frame, mock_model)
    if was_hit:
        hits += 1
    else:
        misses += 1

final_stats = cache3.get_stats()
print(f'  Video sim: {hits} hits, {misses} misses, hit_rate={final_stats["hit_rate"]:.1%}')
print(f'  {cache3}')
print('  PASSED')

print('\n All Frame Cache tests passed!')